# Wikidata Defense Domain Extraction Pipeline

This notebook extracts defense-related entities from a local Wikidata JSON dump (`latest-all.json.bz2`). It identifies entities belonging to defense-related classes (conflicts, military units, weapons, etc.) and builds 2-hop knowledge subgraphs.

### Pipeline Steps
1. Scan Wikidata dump for defense-related entities (35 entity classes)
2. Extract entity properties, labels, descriptions, and Wikipedia links
3. Build 2-hop knowledge subgraphs for each entity
4. Save outputs as JSON, CSV, and summary statistics

### Requirements
```
pip install pandas tqdm
```

### Input
- `latest-all.json.bz2` — Wikidata dump ([download](https://dumps.wikimedia.org/wikidatawiki/entities/))

## Imports

In [ ]:
import json
import bz2
import os
from typing import Dict, List, Optional
from collections import defaultdict
from tqdm import tqdm
import pandas as pd

## Wikidata Defense Entity Extractor

In [ ]:
class WikidataDefenceExtractor:
    """Extract defence-related entities from local Wikidata JSON dump"""
    
    def __init__(self, wikidata_dump_path: str):
        self.wikidata_dump_path = wikidata_dump_path
        
        # Defence-related entity classes (Wikidata Q-IDs)
        self.defence_classes = {
            "Q180684": "conflict",
            "Q178561": "battle", 
            "Q645883": "military operation",
            "Q176799": "military unit",
            "Q18643213": "military equipment",
            "Q728": "weapon",
            "Q177597": "naval vessel",
            "Q197": "aircraft",
            "Q12876": "tank",
            "Q47064": "military personnel",  
            "Q4991371": "soldier", 
            "Q15627509": "military organization",


        # NEW: Defence organizations & agencies
            "Q4508": "navy",
            "Q47913": "intelligence agency",
            "Q917182": "military academy",
            "Q245016": "military base",
            "Q1934969": "defence contractor",



       # NEW: Military structures & ranks
            "Q56019": "military rank",

    
        # NEW: Weapon types (more specific)
            "Q974850": "missile",
            "Q12796": "firearm",
            "Q64418": "artillery",
            "Q1184840": "military vehicle",
            "Q361183": "submarine",
            "Q216916": "military aircraft",
            "Q2031121": "warship",
    
        # NEW: Strategic concepts
            "Q1140224": "military doctrine",
            "Q1127126": "military alliance",
            "Q131569": "treaty",  # often defence-related
    
        # NEW: Defence technology
            "Q329864": "military technology",
            "Q1155630": "missile defense",
            "Q47528": "radar",
    
         # NEW: Military decorations & awards
            "Q1788716": "military decoration",
            "Q1584344": "military medal",     
        }
        
        # Properties we want to extract for relations
        self.important_properties = {
            'P31': 'instance of',
            'P279': 'subclass of',
            'P361': 'part of',
            'P527': 'has part',
            'P155': 'follows',
            'P156': 'followed by',
            'P276': 'location',
            'P17': 'country',
            'P580': 'start time',
            'P582': 'end time',
            'P710': 'participant',
            'P137': 'operator',
            'P176': 'manufacturer',
            'P287': 'designed by',
            'P170': 'creator',


        # NEW: Military-specific properties
            'P241': 'military branch',
            'P607': 'participated in conflict',
            'P1344': 'participant in',
            'P463': 'member of',  # for alliances
            'P410': 'military rank',
            'P598': 'commander of',
            'P3320': 'board member',  # for defence contractors
            # 'P154': 'logo image',
            # 'P18': 'image',
            'P793': 'significant event',
            'P571': 'inception',
            'P576': 'dissolved/abolished',
            'P1449': 'nickname',
            'P2047': 'duration',
            'P1128': 'employees',  # for defence orgs
            'P159': 'headquarters location',
            'P466': 'occupant',  # for military bases



        }
        
    def is_defence_entity(self, entity: Dict) -> Optional[str]:
        """Check if entity belongs to defence domain and return its class"""
        if 'claims' not in entity:
            return None
            
        # Check P31 (instance of) claims
        if 'P31' in entity['claims']:
            for claim in entity['claims']['P31']:
                try:
                    if claim['mainsnak']['snaktype'] == 'value':
                        value = claim['mainsnak']['datavalue']['value']['id']
                        if value in self.defence_classes:
                            return self.defence_classes[value]
                except (KeyError, TypeError):
                    continue
        
        return None
    

    
    def extract_wikipedia_info(self, entity: Dict) -> Dict:
        """Extract Wikipedia page information from entity sitelinks"""
        wikipedia_info = {
            'has_wikipedia': False,
            'wikipedia_url': None,
            'wikipedia_title': None
        }
        
        if 'sitelinks' in entity and 'enwiki' in entity['sitelinks']:
            enwiki = entity['sitelinks']['enwiki']
            title = enwiki.get('title', '')
            
            if title:
                wikipedia_info['has_wikipedia'] = True
                wikipedia_info['wikipedia_title'] = title
                # Construct URL from title
                wikipedia_info['wikipedia_url'] = f"https://en.wikipedia.org/wiki/{title.replace(' ', '_')}"
        
        return wikipedia_info    
  
    
    def extract_entity_data(self, entity: Dict, entity_class: str) -> Dict:
        """Extract relevant data from a Wikidata entity"""
        entity_id = entity.get('id', '')
        
        # Extract labels
        labels = entity.get('labels', {})
        entity_label = labels.get('en', {}).get('value', '') if labels else ''
        
        # Extract descriptions
        descriptions = entity.get('descriptions', {})
        entity_description = descriptions.get('en', {}).get('value', '') if descriptions else ''
        
        # Extract all claims/properties
        all_properties = {}
        if 'claims' in entity:
            for prop_id, claims in entity['claims'].items():
                prop_values = []
                for claim in claims:
                    try:
                        if claim['mainsnak']['snaktype'] == 'value':
                            datavalue = claim['mainsnak']['datavalue']
                            if datavalue['type'] == 'wikibase-entityid':
                                prop_values.append({
                                    'type': 'entity',
                                    'id': datavalue['value']['id']
                                })
                            elif datavalue['type'] == 'string':
                                prop_values.append({
                                    'type': 'string',
                                    'value': datavalue['value']
                                })
                            elif datavalue['type'] == 'time':
                                prop_values.append({
                                    'type': 'time',
                                    'value': datavalue['value']['time']
                                })
                    except (KeyError, TypeError):
                        continue
                
                if prop_values:
                    all_properties[prop_id] = prop_values

        # Extract Wikipedia information
        wikipedia_info = self.extract_wikipedia_info(entity)


        return {
            'entity_id': entity_id,
            'entity_label': entity_label,
            'entity_description': entity_description,
            'entity_class': entity_class,
            'properties': all_properties,
            'aliases': [alias.get('value', '') for alias in entity.get('aliases', {}).get('en', [])],
            'has_wikipedia': wikipedia_info['has_wikipedia'],
            'wikipedia_url': wikipedia_info['wikipedia_url'],
            'wikipedia_title': wikipedia_info['wikipedia_title']
        }
    
    def extract_relations(self, entity_data: Dict) -> List[Dict]:
        """Extract entity-to-entity relations from properties"""
        relations = []
        properties = entity_data.get('properties', {})
        
        for prop_id, values in properties.items():
            prop_name = self.important_properties.get(prop_id, prop_id)
            
            for value in values:
                if value.get('type') == 'entity':
                    relations.append({
                        'source_id': entity_data['entity_id'],
                        'property': prop_name,
                        'property_id': prop_id,
                        'target_id': value['id']
                    })
        
        return relations
    
    def process_dump(self, max_entities: Optional[int] = None) -> tuple:
        """Process Wikidata dump and extract defence entities"""
        print(f"\n{'='*60}")
        print("PROCESSING LOCAL WIKIDATA DUMP")
        print(f"{'='*60}")
        print(f"Reading from: {self.wikidata_dump_path}")
        
        entities = []
        all_relations = []
        entity_class_counts = defaultdict(int)
        processed_count = 0
        
        # Check file size
        file_size = os.path.getsize(self.wikidata_dump_path)
        print(f"File size: {file_size / (1024**3):.2f} GB")
        print("\nProcessing entities (this may take a while)...")
        
        try:
            with bz2.open(self.wikidata_dump_path, 'rt', encoding='utf-8') as f:
                # Skip opening bracket
                first_line = f.readline()
                
                with tqdm(desc="Scanning entities", unit=" entities") as pbar:
                    for line in f:
                        # Skip empty lines and closing bracket
                        line = line.strip()
                        if not line or line == ']':
                            continue
                        
                        # Remove trailing comma if present
                        if line.endswith(','):
                            line = line[:-1]
                        
                        try:
                            # Parse JSON entity
                            entity = json.loads(line)
                            processed_count += 1
                            pbar.update(1)
                            
                            # Check if this is a defence entity
                            entity_class = self.is_defence_entity(entity)
                            
                            if entity_class:
                                # Extract entity data
                                entity_data = self.extract_entity_data(entity, entity_class)
                                entities.append(entity_data)
                                
                                # Extract relations
                                relations = self.extract_relations(entity_data)
                                all_relations.extend(relations)
                                
                                entity_class_counts[entity_class] += 1
                                
                                # Print progress every 100 defence entities
                                if len(entities) % 10000 == 0:
                                    print(f"\n  Found {len(entities)} defence entities (scanned {processed_count:,} total)")
                                
                                # Stop if max_entities reached
                                if max_entities and len(entities) >= max_entities:
                                    print(f"\n✓ Reached target of {max_entities} defence entities!")
                                    break
                            
                            # Progress update every 1000k entities
                            if processed_count % 1000000 == 0:
                                print(f"\n  Scanned {processed_count:,} entities, found {len(entities)} defence entities")
                        
                        except json.JSONDecodeError:
                            continue
                        except Exception:
                            continue
        
        except Exception as e:
            print(f"\n⚠ Error processing dump: {e}")
            print(f"Recovered {len(entities)} entities before error")
        
        print(f"\n{'='*60}")
        print("EXTRACTION COMPLETE")
        print(f"{'='*60}")
        print(f"Total entities scanned: {processed_count:,}")
        print(f"Defence entities found: {len(entities)}")
        print(f"Relations extracted: {len(all_relations)}")
        print(f"\nEntity distribution by class:")
        for entity_class, count in sorted(entity_class_counts.items(), key=lambda x: -x[1]):
            print(f"  - {entity_class}: {count}")
        
        return entities, all_relations


## Extraction Pipeline

In [ ]:
class WikidataDefencePipeline:
    """Main pipeline for Wikidata defence domain extraction"""
    
    def __init__(self, wikidata_dump_path: str, output_dir: str = "./wikidata_defence_output"):
        self.wikidata_dump_path = wikidata_dump_path
        self.output_dir = output_dir
        self.extractor = WikidataDefenceExtractor(wikidata_dump_path)
        
        self.entities = []
        self.relations = []
        
    def build_2hop_subgraph(self, entity_id: str, entities_dict: Dict) -> Dict:
        """Build 2-hop subgraph for an entity"""
        # Get direct relations (1-hop)
        radius_1 = [r for r in self.relations if r['source_id'] == entity_id]
        

        # Get related entity IDs that are also in our extracted entities
        related_ids = {r['target_id'] for r in radius_1 if r['target_id'] in entities_dict}


        # Get related entity IDs
        # related_ids = {r['target_id'] for r in radius_1}
        
        # Get relations of related entities (2-hop)
        radius_2 = []
        for related_id in list(related_ids)[:10]:  # Limit to 10 to avoid explosion
            related_relations = [r for r in self.relations if r['source_id'] == related_id]
            if related_relations:
                radius_2.append({
                    'entity_id': related_id,
                    'entity_label': entities_dict.get(related_id, {}).get('entity_label', ''),
                    'relations': related_relations[:5]  # Top 5 relations
                })
        
        return {
            'center_entity_id': entity_id,
            'center_entity_label': entities_dict.get(entity_id, {}).get('entity_label', ''),
            'radius_1': radius_1,
            'radius_2': radius_2
        }
    
    def create_unified_dataset(self, subgraphs: Dict) -> List[Dict]:
        """Create unified dataset with all entity information"""
        unified = []
        
        for entity in self.entities:
            entity_id = entity['entity_id']
            
            record = {
                'entity_id': entity_id,
                'entity_label': entity['entity_label'],
                'entity_description': entity['entity_description'],
                'entity_class': entity['entity_class'],
                'aliases': entity['aliases'],
                'properties': entity['properties'],
                'has_wikipedia': entity['has_wikipedia'],
                'wikipedia_url': entity['wikipedia_url'],
                'wikipedia_title': entity['wikipedia_title'],
                'has_subgraph': entity_id in subgraphs,
                'direct_relations_count': len(subgraphs[entity_id]['radius_1']) if entity_id in subgraphs else 0,
                'two_hop_entities_count': len(subgraphs[entity_id]['radius_2']) if entity_id in subgraphs else 0,
            }
            
            unified.append(record)
        
        return unified
    
    def run_pipeline(self, max_entities: Optional[int] = None):
        """Execute the complete Wikidata extraction pipeline"""
        print(f"\n{'='*60}")
        print("WIKIDATA DEFENCE DOMAIN EXTRACTION PIPELINE")
        print(f"{'='*60}")
        
        # Step 1: Extract from Wikidata
        print("\n[STEP 1] Extracting defence entities from Wikidata dump...")
        self.entities, self.relations = self.extractor.process_dump(max_entities)
        
        if not self.entities:
            print("⚠ No entities extracted. Check your Wikidata dump file.")
            return []
        
        # Create entity lookup dictionary
        entities_dict = {e['entity_id']: e for e in self.entities}
        
        # Step 2: Build subgraphs
        print("\n[STEP 2] Building 2-hop knowledge subgraphs...")
        subgraphs = {}
        for entity in tqdm(self.entities, desc="Building subgraphs"):
            entity_id = entity['entity_id']
            subgraphs[entity_id] = self.build_2hop_subgraph(entity_id, entities_dict)
        
        print(f"✓ Created {len(subgraphs)} subgraphs")
        
        # Step 3: Create unified dataset
        print("\n[STEP 3] Creating unified dataset...")
        unified_data = self.create_unified_dataset(subgraphs)
        print(f"✓ Unified dataset created with {len(unified_data)} entities")
        
        # Step 4: Save outputs
        print("\n[STEP 4] Saving outputs...")
        self.save_outputs(unified_data, subgraphs)
        
        print(f"\n{'='*60}")
        print("PIPELINE COMPLETE")
        print(f"{'='*60}")
        
        return unified_data
    
    def save_outputs(self, unified_data: List[Dict], subgraphs: Dict):
        """Save all outputs to files"""
        os.makedirs(self.output_dir, exist_ok=True)
        
        # Save unified dataset (JSON)
        print(f"  Saving unified dataset...")
        with open(f"{self.output_dir}/defence_entities.json", 'w', encoding='utf-8') as f:
            json.dump(unified_data, f, indent=2, ensure_ascii=False)
        
        # Save as CSV (without nested properties)
        print(f"  Saving CSV...")
        csv_data = []
        for entity in unified_data:
            csv_record = {
                'entity_id': entity['entity_id'],
                'entity_label': entity['entity_label'],
                'entity_description': entity['entity_description'],
                'entity_class': entity['entity_class'],
                'aliases': '; '.join(entity['aliases']),
                'has_wikipedia': entity['has_wikipedia'],
                'wikipedia_url': entity['wikipedia_url'],
                'wikipedia_title': entity['wikipedia_title'],
                'has_subgraph': entity['has_subgraph'],
                'direct_relations_count': entity['direct_relations_count'],
                'two_hop_entities_count': entity['two_hop_entities_count'],
            }
            csv_data.append(csv_record)
        
        df = pd.DataFrame(csv_data)
        df.to_csv(f"{self.output_dir}/defence_entities.csv", index=False, encoding='utf-8')
        
        # Save relations
        print(f"  Saving relations...")
        with open(f"{self.output_dir}/entity_relations.json", 'w', encoding='utf-8') as f:
            json.dump(self.relations, f, indent=2, ensure_ascii=False)
        
        # Save subgraphs (sample of 100 for inspection)
        print(f"  Saving subgraph samples...")
        sample_subgraphs = dict(list(subgraphs.items())[:100])
        with open(f"{self.output_dir}/entity_subgraphs_sample.json", 'w', encoding='utf-8') as f:
            json.dump(sample_subgraphs, f, indent=2, ensure_ascii=False)
        

        # Save full subgraphs (all entities)
        print("  Saving full subgraphs...")
        with open(f"{self.output_dir}/entity_subgraphs.json", 'w', encoding='utf-8') as f:
            json.dump(subgraphs, f, indent=2, ensure_ascii=False)



        # Save summary statistics
        print(f"  Generating summary...")
        class_counts = defaultdict(int)
        for e in unified_data:
            class_counts[e['entity_class']] += 1
        
        with open(f"{self.output_dir}/SUMMARY.txt", 'w') as f:
            f.write("="*60 + "\n")
            f.write("WIKIDATA DEFENCE DOMAIN EXTRACTION SUMMARY\n")
            f.write("="*60 + "\n\n")
            f.write(f"Total entities extracted: {len(unified_data)}\n")
            f.write(f"Total relations extracted: {len(self.relations)}\n")
            f.write(f"Entities with subgraphs: {len(subgraphs)}\n\n")
            
            f.write("Entity distribution by class:\n")
            for cls, count in sorted(class_counts.items(), key=lambda x: -x[1]):
                f.write(f"  - {cls}: {count}\n")
            
            f.write(f"\nAverage direct relations per entity: {len(self.relations)/len(unified_data):.2f}\n")
            
            entities_with_relations = sum(1 for e in unified_data if e['direct_relations_count'] > 0)
            f.write(f"Entities with at least one relation: {entities_with_relations} ({entities_with_relations/len(unified_data)*100:.1f}%)\n")
        
        print(f"\n✓ All outputs saved to: {self.output_dir}/")
        print(f"  - defence_entities.json (full data)")
        print(f"  - defence_entities.csv (tabular format)")
        print(f"  - entity_relations.json (all relations)")
        print(f"  - entity_subgraphs_sample.json (sample subgraphs)")
        print(f"  - entity_subgraphs.json (full subgraphs)")
        print(f"  - SUMMARY.txt (statistics)")


## Run Pipeline

In [ ]:
def main():
    """Main execution function"""
    print("\n🚀 WIKIDATA DEFENCE DOMAIN EXTRACTION PIPELINE\n")

    # CONFIGURE YOUR PATHS HERE
    WIKIDATA_DUMP = "./latest-all.json.bz2"  # Path to your local Wikidata dump
    OUTPUT_DIR = "./Wikidata_output"
    
    # Validate input file
    if not os.path.exists(WIKIDATA_DUMP):
        print(f"❌ ERROR: Wikidata dump not found at: {WIKIDATA_DUMP}")
        print("\nPlease download the Wikidata dump from:")
        print("https://dumps.wikimedia.org/wikidatawiki/entities/")
        print("(File: latest-all.json.bz2)")
        return
    
    # Initialize pipeline
    pipeline = WikidataDefencePipeline(
        wikidata_dump_path=WIKIDATA_DUMP,
        output_dir=OUTPUT_DIR
    )
    
    try:
        # Run pipeline
        # Set max_entities to None to extract ALL defence entities
        # Set to a number (e.g., 100) for testing/sampling
        unified_data = pipeline.run_pipeline(
            max_entities=None  # Change this: 10 for testing, None for full extraction
        )
        
        if unified_data:
            print(f"\n✅ SUCCESS!")
            print(f"   Extracted {len(unified_data)} defence entities")
            print(f"   Check '{OUTPUT_DIR}/' for all outputs")
        else:
            print("\n❌ Pipeline completed but no entities were extracted")
            
    except KeyboardInterrupt:
        print("\n\n⚠ Pipeline interrupted by user")
    except Exception as e:
        print(f"\n❌ Pipeline failed with error: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
if __name__ == "__main__":
    main()